# ARE-007A: validation evaluation dashboard

This notebook evaluates saved Popularity and ALS models. No training, API calls, source downloads or test evaluation. K values are configured in configs/evaluation.json (default 5, 10, 15, 20). All inference, metrics, grouping and report serialization live in src/. Saved reports remain under ignored artifacts/evaluation/.

Outputs contain public-dataset validation IDs and recommendations. Clear outputs and execution counts before saving this notebook. It never uses YummyAnime profiles.

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
from anime_recommender.exploration import project_root
from anime_recommender.evaluation.ranking_evaluator import run_evaluation
from anime_recommender.evaluation.dashboard import dashboard_tables
from anime_recommender.evaluation.reports import load_report

root = project_root()
config_path = "configs/evaluation.json"
display_k = 10

## Evaluate saved models and verify saved report

One maximum-K recommendation list is generated per model/user, then reused as prefixes for every K and for Full/Warm. Every observed context item is excluded (including low/zero ratings); only warm positive context goes into fixed-item ALS fold-in. Targets are used only after generation. ALS-only and ALS+fallback are separate systems in the tables. Model factors are checked for immutability.

In [ ]:
report = run_evaluation(root, config_path=config_path)
loaded = load_report(root / report.metadata["report_dir"])
view = dashboard_tables(loaded, k=display_k)
display(loaded.metadata)

## Main Full metrics

All five metrics are undefined for empty relevant target sets. Macro averages exclude those values and report eligible counts. Precision divides by K even for short lists. Per-user AP divides by min(K, number of relevant targets); MAP is its macro mean. DCG is binary with log2(rank+1) discounts, and IDCG includes all relevant targets up to K. ALS scores are ranking values, not probabilities.

The primary curves compare ALS-only with Popularity. The explicit ALS+fallback row can coincide with ALS-only when all users have warm context; this does not measure a fallback improvement.

In [ ]:
display(view["main"].style.format(na_rep="N/A"))

## Recall@K and NDCG@K

In [ ]:
view["recall_curve"].plot(marker="o", title="Full Recall@K on validation")
plt.xlabel("K")
plt.ylabel("Macro Recall")
plt.tight_layout()
plt.show()
view["ndcg_curve"].plot(marker="o", title="Full NDCG@K on validation")
plt.xlabel("K")
plt.ylabel("Macro NDCG")
plt.tight_layout()
plt.show()

## Full vs Warm Recall

Both modes score the same ordered lists. Full retains cold targets in Recall/AP/IDCG denominators. Warm uses only warm targets and excludes users without warm targets, without excluding users based on their context. Unlike legacy ARE-006 warm diagnostics, warm-context availability is not an eligibility requirement here. Users without warm context are still visible in ALS-only, with empty recommendations, or in the separately labeled fallback system.

In [ ]:
view["full_warm_recall"].plot(marker="o", title="Full vs Warm Recall@K")
plt.xlabel("K")
plt.ylabel("Macro Recall")
plt.tight_layout()
plt.show()

## Catalog coverage

In [ ]:
display(loaded.tables["coverage"])
view["catalog_coverage"].plot(marker="o", title="Distinct recommended items / train vocabulary")
plt.xlabel("K")
plt.ylabel("Catalog coverage")
plt.tight_layout()
plt.show()

## Context lengths and cohorts

Cohorts use distinct warm positive context only: 0, 1, 2-4, 5-9, 10-19, 20+. Neither target size nor relevance influences assignment. Empty groups have N/A metrics, zero users and zero eligible users. Warm context coverage is the fraction of positive context interactions belonging to train vocabulary. Cold target share uses all positive targets, even in warm-mode diagnostic tables.

Target-size mean and median use every cohort user, including zero-target users. Both full and warm target counts are descriptive diagnostics computed after cohort assignment. More targets mechanically lower Recall for the same number of hits; compare these sizes before interpreting short/long-history differences.

In [ ]:
display(loaded.tables["history_distribution"])
display(loaded.tables["users"]["warm_context"].describe())
loaded.tables["users"]["warm_context"].plot.hist(bins=20, title="Validation warm positive context lengths")
plt.xlabel("Distinct warm positive context items")
plt.ylabel("Users")
plt.tight_layout()
plt.show()
display(view["cohorts"].style.format(na_rep="N/A"))
display(view["cohort_target_sizes"].style.format(na_rep="N/A"))
view["cohort_target_sizes"][["positive_targets_mean", "positive_targets_median"]].plot.bar(
    title="Full positive target sizes by context cohort (all users)")
plt.xlabel("Warm positive context cohort")
plt.ylabel("Positive targets per user")
plt.tight_layout()
plt.show()
view["cohort_recall"].plot.bar(title=f"Full Recall@{display_k} by context cohort")
plt.xlabel("Warm positive context cohort")
plt.ylabel("Macro Recall")
plt.tight_layout()
plt.show()

## Cold items, context availability and fallback

In [ ]:
display(view["cold_statistics"].T)
display(loaded.tables["statuses"].head(12))

## Successful and unsuccessful ALS validation examples

At K=10 (or the first configured K if 10 is absent), choose the smallest validation user ID with a hit and the smallest with positive targets but no hit. Examples are selected after predictions, not used to train/tune anything. A missing type means no matching user exists; unavailable ALS can have an empty recommendation table. Titles come from the certified local training catalog. is_hit is post-hoc target membership.

In [ ]:
display(view["example_users"])
display(view["examples"])

## Limits

This is fixed-model evaluation on a small sample with random context/target splitting and no timestamps. It measures preference reconstruction rather than future chronological behavior. No confidence intervals, significance claims or hyperparameter tuning are implemented. Catalog exposure is not a complex diversity metric. ARE-007B is not started. See docs/ARE-007A-evaluation-framework.md for definitions, reproducibility checks and comparison with ARE-006.